# Is Dar es Salaam warmer than forty years ago?

**MATH4SDG / UDSM — optional project, not a compulsory lesson**

**Research question:** Has the *NASA POWER estimated* annual mean two-metre air temperature of the grid cell containing Dar es Salaam increased over 1986–2025? How confident should we be?

**Crucial distinction:** NASA POWER T2M is a **gridded meteorological reanalysis estimate** (primarily based on MERRA-2), not a direct thermometer record from Dar es Salaam Airport. It can have excellent temporal coverage while still having model biases, spatial averaging and changes in assimilated observations. See [official methodology](https://power.larc.nasa.gov/docs/methodology/data/sources_meteorology/).

**Source selection:** The notebook uses a locally downloaded NASA POWER CSV **if present**. Otherwise it uses an explicitly labeled **SYNTHETIC** dataset only for code practice. Synthetic output must NEVER be reported as evidence about real climate.

**Before the bootcamp, while online**, download using:

```sh
uv run python tools/download_nasa_power.py --location dar_es_salaam
```

The private CSV and provenance metadata are excluded from Git. See [project README](../projects/nasa-power-temperature/README.md) for details.

This activity adapts ideas from the [MAT102 Meteostat notebooks](https://github.com/mbr085/V26MAT102) but gives data-source quality, statistical uncertainty and interpretation equal importance to fitting a model. Code: MIT; teaching prose: CC BY 4.0; datasets: separate terms.

## 1. Make the scientific question precise

A single hot day says little about long-term temperature change. We will estimate a **descriptive linear slope** in °C/decade from annual temperature summaries, then compare the first and last ten years.

We are studying the POWER **grid-cell estimate**, not direct temperature measurements at a specific airport, nor the whole country. What additional evidence would justify a statement about Dar es Salaam residents' experienced climate, or about East Africa?

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from tools.generate_synthetic_temperature import generate
from tools.temperature_methods import annual_means, estimate_trend

REAL = ROOT / "data" / "private" / "nasa_power_dar_es_salaam_1986_2025.csv"
if REAL.exists():
    data = pd.read_csv(REAL, parse_dates=["date"])
    SOURCE = "REAL NASA POWER T2M GRIDDED REANALYSIS (NOT STATION MEASUREMENTS)"
else:
    data = generate()
    SOURCE = "SYNTHETIC DATA ONLY — ARTIFICIAL EXAMPLE, NOT CLIMATE EVIDENCE"
print("=" * 72)
print(SOURCE)
print("=" * 72)
data = data.sort_values("date")
if data["date"].duplicated().any():
    raise ValueError("Repeated dates in the input; resolve provenance before analysis")
data["temp"] = pd.to_numeric(data["temp"], errors="coerce")
data = data.set_index("date").loc["1986":"2025"]
if data.empty:
    raise ValueError("No observations or estimates for requested 1986–2025 period")
if REAL.exists():
    metadata = REAL.with_suffix(".json")
    print("Provenance metadata:", metadata if metadata.exists() else "MISSING (re-download recommended)")
print("Records:", len(data), "Days with values:", int(data["temp"].notna().sum()))
display(data.head())

## 2. Data completeness is not scientific validation

First check missing dates, `NaN`, and the distribution of missingness. NASA POWER estimates normally have much better historical coverage than the Meteostat station record examined for this project. But **complete reanalysis is not the same as complete station observations**.

A previous day's temperature is not today's temperature: we must not quietly forward-fill missing months. Compare the actual values of coverage before believing a regression estimate.

In [ ]:
daily = data["temp"].asfreq("D")
coverage = daily.notna().groupby(daily.index.year).mean()
print("Missing days:", int(daily.isna().sum()), "out of", len(daily))
print("Lowest coverage years:")
print(coverage.nsmallest(8).round(3))
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
daily.iloc[:730].plot(ax=axes[0], title="First two years (gaps left unfilled)")
axes[0].set_ylabel("°C")
coverage.plot(ax=axes[1], marker="o", title="Fraction of days with temperature by year")
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("Coverage")
plt.tight_layout()
plt.show()

## 3. Seasonal variation

Plot the mean temperature by calendar month. Seasonal variation may dominate the short-term record. Why is a comparison of the same months in different years more meaningful than a comparison of individual days?

Reanalysis temperatures represent a spatial grid cell. Even the shape of the seasonal cycle should ideally be compared with trusted observations.

In [ ]:
seasonal = daily.groupby(daily.index.month).mean()
ax = seasonal.plot(marker="o", figsize=(8, 3), title="Typical seasonal temperature")
ax.set(xlabel="Calendar month", ylabel="°C")
plt.show()

## 4. Annual summaries and regression

We summarize daily values into monthly means, require **at least 20 valid days in each month**, and average the 12 months equally. Report any excluded years. For NASA POWER the series may be complete, but the exact coverage **must be checked**, not assumed.

For eligible years, fit

$$T_y = \alpha + \beta(y-y_0) + \varepsilon_y.$$

Here $10\beta$ has units °C per decade. What does this slope measure? Does a linear trend necessarily capture all the features of the data?

In [ ]:
annual, monthly_table = annual_means(daily, min_days_per_month=20)
print(f"Eligible years: {len(annual)} of {len(monthly_table)}")
print("Excluded:", sorted(set(monthly_table.index) - set(annual.index)))
if len(annual) < 15:
    raise ValueError("Too few complete years: investigate the data response, grid cell, variable or time period.")
ax = annual.plot(marker="o", figsize=(10, 4), title="Annual mean: gridded estimates (or labeled synthetic data)")
ax.set(xlabel="Year", ylabel="°C")
plt.show()

## 5. Least squares with NumPy

For $n$ eligible annual summaries, least squares minimizes $\sum_i (y_i-\alpha-\beta t_i)^2$.

Ask an AI assistant to explain the two columns of the regression design matrix. Then verify its answer. The calculation below uses `np.linalg.lstsq` and checks the slope independently with SciPy. Models should remain understandable even when AI writes some of their code.

In [ ]:
results = estimate_trend(annual, maxlags=3)
print(f"Descriptive slope: {results['slope_c_decade']:.3f} °C/decade")
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(annual.index, annual.values, s=24, label="Annual summaries")
ax.plot(results["years"], results["fitted"], label="Least-squares line")
ax.set(xlabel="Year", ylabel="°C", title="POWER grid-cell trend (or explicitly synthetic example)")
ax.legend()
plt.show()

## 6. Statistical intervals versus scientific confidence

Compare the ordinary least-squares confidence interval with a **HAC (heteroskedasticity/autocorrelation-consistent)** interval using three annual lags. Both depend on assumptions about the time series; neither includes uncertainty caused by the reanalysis system, changes in its observational inputs, or the mismatch between a grid cell and a point observation.

A 95% confidence interval is **not** a 95% probability statement about this particular fixed trend parameter. A small $p$-value is not the probability the null hypothesis is true.

If you obtain a narrow interval, what evidence could still make the overall climate conclusion unreliable?

In [ ]:
print("Slope (°C/decade):", round(results["slope_c_decade"], 3))
print("Conventional 95% interval:", np.round(results["conventional_ci_c_decade"], 3))
print("HAC 95% interval, annual maxlags=3:", np.round(results["hac_ci_c_decade"], 3))
print("Lag-one correlation of annual residuals:", round(results["residual_lag1"], 3))
print("Reminder:", SOURCE)

## 7. Compare the earliest and latest decades

If at least eight adequately covered years are present in each decade, compare **1986–1995** with **2016–2025**. This answers a different question from the slope of a fitted straight line. If the data do not meet the minimum, we should report that rather than impute years.

Even a convincing difference concerns the **reanalysis grid-cell series**; separate corroboration is needed for an observational or regional warming claim.

In [ ]:
early = annual.loc[1986:1995]
late = annual.loc[2016:2025]
print("Early decade eligible years:", len(early), "/ 10")
print("Late decade eligible years:", len(late), "/ 10")
if len(early) >= 8 and len(late) >= 8:
    print(f"Recent minus early decade mean: {late.mean() - early.mean():.3f} °C")
else:
    print("Insufficient coverage for a reliable decade comparison.")

## 8. Does the estimated warming depend on the starting year?

**Our central follow-up question:** How much confidence should we place in the estimated trend if the answer changes when we choose a different starting year?

Fit separate lines to annual temperature summaries for **1986–2025, 1991–2025, 1996–2025, and 2001–2025**. Each fit uses a different span of the *same* NASA POWER reanalysis series. Calculate the slope in °C per decade, an interval allowing for possible annual autocorrelation (HAC, with three annual lags), and the number of eligible years.

- Do you see a systematic change in the estimated slopes?
- Can a 40-year positive slope coexist with a small slope over the final 25 years?
- Does a small slope mean that warming has stopped? What does its confidence interval say?
- The windows **overlap**. Why is comparing two individual confidence intervals *not* a valid significance test for their difference?
- What would count as evidence that the trend itself has changed, rather than that a shorter fit is more sensitive to variability?

Do not pick the start year that gives the answer you prefer. Any inference about climate is conditional on the reanalysis data and statistical assumptions; this is **not** direct evidence about a particular weather station.

In [ ]:
# A sensitivity analysis: fit each nested window independently.
# Note: these fitted slopes are CORRELATED because the windows overlap.
sensitivity = []
for first in [1986, 1991, 1996, 2001]:
    part = annual.loc[first:2025]
    if len(part) < 15:
        continue
    fit = estimate_trend(part, maxlags=3)
    lo, hi = fit["hac_ci_c_decade"]
    sensitivity.append({
        "start_year": first,
        "eligible_years": len(part),
        "slope_C_per_decade": fit["slope_c_decade"],
        "HAC_95_lower": lo,
        "HAC_95_upper": hi,
    })

sensitivity_table = pd.DataFrame(sensitivity)
if sensitivity_table.empty:
    raise ValueError("Not enough eligible annual summaries for the sensitivity analysis")
display(sensitivity_table.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = sensitivity_table["start_year"].to_numpy()
y = sensitivity_table["slope_C_per_decade"].to_numpy()
lower = sensitivity_table["HAC_95_lower"].to_numpy()
upper = sensitivity_table["HAC_95_upper"].to_numpy()
axes[0].errorbar(x, y, yerr=[y - lower, upper - y], fmt="o", capsize=4)
axes[0].axhline(0, color="gray", linestyle="--")
axes[0].set(
    title="Nested periods ending in 2025 (HAC 95% intervals)",
    xlabel="Start year", ylabel="Fitted slope (°C/decade)"
)
axes[0].set_xticks(x)
axes[1].plot(results["years"], annual.values - results["fitted"], marker="o")
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set(
    title="Residuals of the full-period line",
    xlabel="Year", ylabel="Residual (°C)"
)
plt.tight_layout()
plt.show()
print("NOTE:", SOURCE)
print("These confidence intervals are NOT a test of the difference between overlapping slopes.")

### Optional investigation: is there evidence of a change in slope?

Comparing the individual intervals is not a valid test of whether the *slopes differ*, because the data windows overlap. An advanced group could explore a **single continuous segmented-regression model** with a candidate change point at 2001:

$$
T_y = \\alpha + \\beta (y-1986) + \\gamma\\max(0,y-2001) + \\varepsilon_y.
$$

Here the model's slope is $\\beta$ before 2001 and $\\beta+\\gamma$ after 2001. Investigate the hypothesis $\\gamma=0$ with a confidence interval that accounts for temporal dependence.

**Important:** This tests a different model and question from comparing the two nested least-squares slopes above. Choosing 2001 because it gave an interesting result and then treating it as a pre-specified change point would overstate the evidence. A convincing investigation should consider sensitivity to the choice of change point, nonlinearity, and uncertainty about the reanalysis source.

**Other challenges:** Re-run annual aggregation with 15 or 25 valid days per month; ask an AI assistant to critique three assumptions, then verify its criticisms; inspect [NASA POWER methodology](https://power.larc.nasa.gov/docs/methodology/data/sources_meteorology/) or the optional [Meteostat missingness exercise](meteostat_station_missingness.ipynb) to discuss data provenance.

**Group report:** State the precise estimand, full-period estimate and interval, starting-year sensitivity, what can and cannot be inferred about a change in trend, source limitations, and a proposed next step. It is acceptable to conclude that the data do not justify a particular claim. **Never present synthetic results as actual observations.**

### Extensions

- Compare the POWER grid cells for Dar es Salaam and Dodoma, using separate downloads and accounting for spatial smoothing.
- Fit monthly temperature anomalies or seasonal sine/cosine regression, then compare with the annual analysis.
- Consider climate mechanisms and independent observations, but do **not** infer causal attribution from a single fitted trend.
- Use the optional Meteostat station record to study why a time series can be unsuitable even when the regression routine happily returns a slope.

Sources: [NASA POWER daily API](https://power.larc.nasa.gov/docs/services/api/temporal/daily/), [data sources](https://power.larc.nasa.gov/docs/methodology/data/sources_meteorology/), [referencing guide](https://power.larc.nasa.gov/docs/referencing/).